In [1]:
from datasets import load_dataset

dataset = load_dataset("ccosme/FiReCS")
print(dataset)

c:\Anaconda\envs\firecs\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 3147/3147 [00:00<00:00, 299362.12 examples/s]

DatasetDict({
    train: Dataset({
        features: ['review', 'label'],
        num_rows: 7340
    })
    test: Dataset({
        features: ['review', 'label'],
        num_rows: 3147
    })
})


In [2]:
print(dataset["train"][0])
print(dataset["train"].features)

{'review': 'im very disappointed kasi di gumana ang dalawa kung order', 'label': 0.0}
{'review': Value('string'), 'label': Value('float64')}


In [3]:
for split in dataset:
    print(split, len(dataset[split]))

import pandas as pd
train_df = dataset["train"].to_pandas()
print(train_df["label"].value_counts())

train 7340
test 3147
label
1.0    2549
2.0    2410
0.0    2381
Name: count, dtype: int64


In [1]:
import sys
sys.path.append("../src")

from preprocessing import clean_text, preprocess_dataframe, stratified_split

# quick test on a made-up example
test_str = "OMGGGGG ang gandaaaaa!!! super satisfied ako..."
print(clean_text(test_str))

OMGG ang gandaa super satisfied ako


In [2]:
from datasets import load_dataset

# Reload if needed (skip if 'dataset' variable already exists from before)
dataset = load_dataset("ccosme/FiReCS")

# Convert HuggingFace splits to pandas
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

print("Raw train shape:", train_df.shape)
print("Raw test shape:", test_df.shape)

c:\Anaconda\envs\firecs\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Raw train shape: (7340, 2)
Raw test shape: (3147, 2)


In [3]:
from preprocessing import preprocess_dataframe, stratified_split

# Clean the text
train_df = preprocess_dataframe(train_df, text_col="review")
test_df = preprocess_dataframe(test_df, text_col="review")

# Stratified train/val split (10% of train)
train_final, val_final = stratified_split(train_df, label_col="label", val_size=0.10, seed=42)

print("Final train shape:", train_final.shape)
print("Val shape:", val_final.shape)
print("Test shape:", test_df.shape)

Final train shape: (6606, 3)
Val shape: (734, 3)
Test shape: (3147, 3)


In [4]:
print(train_final[["review", "clean_review"]].head(5))

                                                 review  \
511   sobrang tagal ng delivery more than 10 days hn...   
5755  wrong size naman binigay nyo order size 31 siz...   
2879  super thank you sa seller for the consideratio...   
6485  hindi man lang pinacked ng maayos sana next ti...   
5244  mesh yung upper part ng shoes mabilis si selle...   

                                           clean_review  
511   sobrang tagal ng delivery more than 10 days hn...  
5755  wrong size naman binigay nyo order size 31 siz...  
2879  super thank you sa seller for the consideratio...  
6485  hindi man lang pinacked ng maayos sana next ti...  
5244  mesh yung upper part ng shoes mabilis si selle...  


In [5]:
import os

os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

# Save raw (untouched original) copies
train_df.to_csv("../data/raw/train_raw.csv", index=False)
test_df.to_csv("../data/raw/test_raw.csv", index=False)

# Save processed (cleaned + split) copies
train_final.to_csv("../data/processed/train.csv", index=False)
val_final.to_csv("../data/processed/val.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)

print("All files saved.")

All files saved.


In [6]:
from feature_extraction import load_fasttext_model, get_word_vector, get_review_vectors

ft_model = load_fasttext_model("tl")
print("Model loaded. Vector size:", ft_model.get_dimension())

 (100.00%) [==================================================>]                                                 ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]>                                                  ]=>                                                 ]=>                                                 ]=>                                                 ]=>

In [7]:
# Sanity check - test on real + hybrid Taglish words
test_words = ["ganda", "nagorder", "satisfied", "icheck"]

for word in test_words:
    vec = get_word_vector(ft_model, word)
    print(word, "-> shape:", vec.shape, "| first 5 values:", vec[:5])

ganda -> shape: (300,) | first 5 values: [ 0.02025932 -0.04195533 -0.0733071   0.03289375 -0.00177784]
nagorder -> shape: (300,) | first 5 values: [-0.01119184 -0.00301755 -0.01868965  0.01917825 -0.01820293]
satisfied -> shape: (300,) | first 5 values: [-0.01698109  0.02575917  0.0310847   0.0268891  -0.03963071]
icheck -> shape: (300,) | first 5 values: [-0.01358646  0.00869025 -0.00623433  0.01999502 -0.01183912]
